In [2]:
# Step - 1
from google.colab import files
uploaded = files.upload()

import pandas as pd, numpy as np, ast
from collections import Counter

fname = list(uploaded.keys())[0]
df = pd.read_csv(fname, low_memory=False)
print(df.shape)

df['created_datetime'] = pd.to_datetime(df['created_datetime'], format='mixed', utc=True)
df['hour_ist'] = df['created_datetime'].dt.tz_convert('Asia/Kolkata').dt.hour
df['vt_list'] = df['violation_type'].apply(ast.literal_eval)

c = Counter()
for l in df['vt_list']:
    for v in l: c[v] += 1
print(pd.Series(c).sort_values(ascending=False).head(10))

print(df['validation_status'].value_counts(dropna=False))
valid = df[~df['validation_status'].isin(['rejected','duplicate'])].copy()
print(f"confirmed: {len(valid)} of {len(df)}")

Saving jan to may police violation_anonymized791b166.csv to jan to may police violation_anonymized791b166.csv
(298450, 24)
WRONG PARKING                                164977
NO PARKING                                   139050
PARKING IN A MAIN ROAD                        23943
DEFECTIVE NUMBER PLATE                         7848
PARKING ON FOOTPATH                            3757
PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC        2403
DOUBLE PARKING                                 2037
PARKING NEAR ROAD CROSSING                     1687
REFUSE TO GO FOR HIRE                           887
PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS       525
dtype: int64
validation_status
NaN           125254
approved      115400
rejected       49754
created1        7044
processing       678
duplicate        320
Name: count, dtype: int64
confirmed: 248376 of 298450


In [3]:
#Step - 2
from sklearn.cluster import DBSCAN

coords = np.radians(valid[['latitude','longitude']].values)
eps_rad = (75/1000)/6371.0088
db = DBSCAN(eps=eps_rad, min_samples=40, metric='haversine', algorithm='ball_tree', n_jobs=-1)
valid['cluster'] = db.fit_predict(coords)

n_clusters = len(set(valid['cluster'])) - (1 if -1 in valid['cluster'].values else 0)
noise = (valid['cluster']==-1).sum()
print(f"clusters: {n_clusters}, noise: {noise} ({100*noise/len(valid):.1f}%)")

clusters: 325, noise: 21928 (8.8%)


In [4]:
# Step - 3
severity = {
    'DOUBLE PARKING': 2.0, 'PARKING NEAR ROAD CROSSING': 2.0,
    'PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS': 2.0,
    'PARKING OPPOSITE TO ANOTHER PARKED VEHICLE': 2.0,
    'PARKING IN A MAIN ROAD': 1.5, 'PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC': 1.3,
    'PARKING OTHER THAN BUS STOP': 1.3, 'PARKING ON FOOTPATH': 1.1,
    'WRONG PARKING': 1.0, 'NO PARKING': 1.0,
}
vehicle_w = {
    'SCOOTER':0.5,'MOTOR CYCLE':0.5,'MOPED':0.5,
    'CAR':1.0,'PASSENGER AUTO':1.0,'VAN':1.0,
    'MAXI-CAB':2.0,'LGV':2.0,'GOODS AUTO':2.0,'PRIVATE BUS':2.0
}
valid['sev_w'] = valid['vt_list'].apply(lambda l: max(severity.get(v,0.8) for v in l))
valid['veh_w'] = valid['vehicle_type'].map(vehicle_w).fillna(1.0)
valid['weighted'] = valid['sev_w']*valid['veh_w']
valid['peak_hr'] = valid['hour_ist'].between(8,13)

clustered = valid[valid.cluster!=-1]
agg = clustered.groupby('cluster').agg(
    n=('id','count'), lat=('latitude','mean'), lon=('longitude','mean'),
    weighted_sum=('weighted','sum'), peak_share=('peak_hr','mean'),
    days_active=('created_datetime', lambda x: x.dt.date.nunique())
).reset_index()

agg['vol_n']  = agg['n']/agg['n'].max()
agg['wt_n']   = agg['weighted_sum']/agg['weighted_sum'].max()
agg['peak_n'] = agg['peak_share']/agg['peak_share'].max()
agg['days_n'] = agg['days_active']/agg['days_active'].max()
agg['impact_score'] = (0.40*agg['wt_n'] + 0.30*agg['vol_n'] + 0.15*agg['peak_n'] + 0.15*agg['days_n'])*100
agg = agg.sort_values('impact_score', ascending=False).reset_index(drop=True)

print(f"{len(agg)} hotspots | top score: {agg.impact_score.max():.1f}")
agg.head(10)

325 hotspots | top score: 92.2


,cluster,n,lat,lon,weighted_sum,peak_share,days_active,vol_n,wt_n,peak_n,days_n,impact_score
0,2,49795,12.972104,77.577592,44066.90,0.478341,152,1.000000,1.000000,0.478341,1.000000,92.175118
1,3,19865,12.981954,77.608066,16883.95,0.727460,151,0.398936,0.383144,0.727460,0.993421,53.107034
2,31,8804,13.010307,77.553295,7836.90,0.615175,151,0.176805,0.177841,0.615175,0.993421,36.546725
3,34,8323,13.000538,77.571070,7205.20,0.564220,152,0.167145,0.163506,0.564220,1.000000,35.017891
4,5,8225,12.999149,77.549239,5942.10,0.483161,152,0.165177,0.134843,0.483161,1.000000,32.596441
5,53,5125,13.071030,77.588417,3053.55,0.805463,136,0.102922,0.069294,0.805463,0.894737,31.362403
6,9,8540,12.933344,77.690135,7272.40,0.360422,129,0.171503,0.165031,0.360422,0.848684,29.882917
7,60,6453,12.974526,77.547241,3725.00,0.495738,150,0.129591,0.084531,0.495738,0.986842,29.507670
8,23,4031,13.185442,77.680109,5803.00,0.364922,147,0.080952,0.131686,0.364922,0.967105,27.676409
9,27,3653,12.967262,77.537826,2596.55,0.543936,148,0.073361,0.058923,0.543936,0.973684,27.322050


In [5]:
# Step - 4
!pip install folium -q
import folium
from collections import Counter

m = folium.Map(location=[12.97, 77.59], zoom_start=11, tiles='cartodbpositron')

for _, r in agg.head(25).iterrows():
    sub = valid[valid.cluster==int(r.cluster)]
    jn = sub.junction_name.value_counts()
    top_jn = jn.index[0] if jn.index[0]!='No Junction' or len(jn)==1 else (jn.index[1] if len(jn)>1 else 'No Junction')
    loc = sub['location'].dropna().str.split(',').str[0].value_counts()
    label = top_jn.split(' - ')[-1] if ' - ' in top_jn else (loc.index[0] if len(loc) else top_jn)
    color = 'red' if r.impact_score>=50 else 'orange' if r.impact_score>=30 else 'blue'
    folium.CircleMarker(
        location=[r.lat, r.lon],
        radius=max(6,(r.n**0.5)/3),
        popup=f"<b>{label}</b><br>Score: {r.impact_score:.1f}<br>Violations: {int(r.n)}",
        color=color, fill=True, fill_opacity=0.5
    ).add_to(m)

m

# File Download
m.save('heatmap.html')
from google.colab import files
files.download('heatmap.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# Step - 5
!pip install folium -q
import folium

m = folium.Map(location=[12.97, 77.59], zoom_start=11, tiles='cartodbpositron')

for _, r in agg.head(25).iterrows():
    sub = valid[valid.cluster==int(r.cluster)]
    jn = sub.junction_name.value_counts()
    top_jn = jn.index[0] if jn.index[0]!='No Junction' or len(jn)==1 else (jn.index[1] if len(jn)>1 else 'No Junction')
    loc = sub['location'].dropna().str.split(',').str[0].value_counts()
    label = top_jn.split(' - ')[-1] if ' - ' in top_jn else (loc.index[0] if len(loc) else top_jn)
    color = 'red' if r.impact_score>=50 else 'orange' if r.impact_score>=30 else 'blue'
    folium.CircleMarker(
        location=[r.lat, r.lon], radius=max(6,(r.n**0.5)/3),
        popup=f"<b>{label}</b><br>Score: {r.impact_score:.1f}<br>Violations: {int(r.n)}",
        color=color, fill=True, fill_opacity=0.5
    ).add_to(m)

m

In [7]:
# Step - 6
rows=[]
for _,r in agg.head(15).iterrows():
    sub = valid[valid.cluster==int(r.cluster)]
    jn = sub.junction_name.value_counts()
    top_jn = jn.index[0] if jn.index[0]!='No Junction' or len(jn)==1 else (jn.index[1] if len(jn)>1 else 'No Junction')
    loc = sub['location'].dropna().str.split(',').str[0].value_counts()
    label = top_jn.split(' - ')[-1] if ' - ' in top_jn else (loc.index[0] if len(loc) else top_jn)
    vt = Counter()
    for l in sub.vt_list:
        for v in l: vt[v]+=1
    top_vt = ', '.join([v for v,_ in vt.most_common(2)])
    hrs = sub.hour_ist.value_counts().sort_values(ascending=False)
    patrol = f"{hrs.index[0]:02d}:00-{hrs.index[0]+1:02d}:00 IST"
    rows.append({'rank':len(rows)+1,'zone':label,'lat':round(r.lat,4),'lon':round(r.lon,4),
        'violations':int(r.n),'score':round(r.impact_score,1),'peak_share_pct':round(100*r.peak_share,1),
        'top_violations':top_vt,'patrol_window':patrol,'days_active':int(r.days_active)})

enforcement_priority = pd.DataFrame(rows)
enforcement_priority

,rank,zone,lat,lon,violations,score,peak_share_pct,top_violations,patrol_window,days_active
0,1,KR Market Junction,12.9721,77.5776,49795,92.2,47.8,"WRONG PARKING, NO PARKING",09:00-10:00 IST,152
1,2,Safina Plaza Junction,12.9820,77.6081,19865,53.1,72.7,"WRONG PARKING, NO PARKING",10:00-11:00 IST,151
2,3,"10th Cross, Dr. Rajkumar Road",13.0103,77.5533,8804,36.5,61.5,"WRONG PARKING, NO PARKING",10:00-11:00 IST,151
3,4,Anand Rao Junction,13.0005,77.5711,8323,35.0,56.4,"WRONG PARKING, NO PARKING",10:00-11:00 IST,152
4,5,Modi Bridge Junction,12.9991,77.5492,8225,32.6,48.3,"WRONG PARKING, NO PARKING",08:00-09:00 IST,152
5,6,Sahakar Nagar Road,13.0710,77.5884,5125,31.4,80.5,"WRONG PARKING, NO PARKING",11:00-12:00 IST,136
6,7,New Horizon College Road,12.9333,77.6901,8540,29.9,36.0,"NO PARKING, WRONG PARKING",04:00-05:00 IST,129
7,8,Hosahalli Metro Station,12.9745,77.5472,6453,29.5,49.6,"NO PARKING, WRONG PARKING",04:00-05:00 IST,150
8,9,Unnamed Road,13.1854,77.6801,4031,27.7,36.5,"NO PARKING, WRONG PARKING",04:00-05:00 IST,147
9,10,"5th Main Road, RPC Layout",12.9673,77.5378,3653,27.3,54.4,"WRONG PARKING, NO PARKING",10:00-11:00 IST,148


In [8]:
# Step - 7
enforcement_priority.to_csv('enforcement_priority.csv', index=False)
agg.to_csv('hotspots_full.csv', index=False)

from google.colab import files
files.download('enforcement_priority.csv')
files.download('hotspots_full.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
# Step - 8
from scipy import stats

zone_density = agg['n'].values
peak_share = agg['peak_share'].values

r_val, p_val = stats.pearsonr(zone_density, peak_share)
print(f"Pearson correlation (violation density vs peak-hour share): r = {r_val:.3f}, p = {p_val:.4f}")

top10 = agg.head(10)
top10_peak_avg = top10['peak_share'].mean()
all_peak_avg = agg['peak_share'].mean()
lift = ((top10_peak_avg - all_peak_avg) / all_peak_avg) * 100

print(f"Top-10 zones peak-hour share : {100*top10_peak_avg:.1f}%")
print(f"All-zones average             : {100*all_peak_avg:.1f}%")
print(f"Congestion-time lift          : +{lift:.1f}%")
print(f"\nHeadline stat for concept note:")
print(f"325 hotspot clusters identified; top-10 zones show {100*top10_peak_avg:.1f}% peak-hour concentration vs {100*all_peak_avg:.1f}% baseline — a {lift:.1f}% congestion-time lift.")

Pearson correlation (violation density vs peak-hour share): r = 0.028, p = 0.6114
Top-10 zones peak-hour share : 54.4%
All-zones average             : 46.8%
Congestion-time lift          : +16.2%

Headline stat for concept note:
325 hotspot clusters identified; top-10 zones show 54.4% peak-hour concentration vs 46.8% baseline — a 16.2% congestion-time lift.
